In [6]:
import polars as pl
from path_config import PathConfig
paths = PathConfig()

In [ ]:
dl_data = pl.read_excel(paths.dl_path, sheet_name='Series Formatted Data')
dl_data = dl_data.drop(["Message", "Technology_Mode"])

Could not determine dtype for column 10, falling back to string
Could not determine dtype for column 11, falling back to string
Could not determine dtype for column 12, falling back to string
Could not determine dtype for column 13, falling back to string
Could not determine dtype for column 15, falling back to string
Could not determine dtype for column 16, falling back to string
Could not determine dtype for column 17, falling back to string
Could not determine dtype for column 18, falling back to string
Could not determine dtype for column 20, falling back to string
Could not determine dtype for column 21, falling back to string
Could not determine dtype for column 22, falling back to string
Could not determine dtype for column 23, falling back to string
Could not determine dtype for column 24, falling back to string
Could not determine dtype for column 27, falling back to string
Could not determine dtype for column 28, falling back to string
Could not determine dtype for column 29,

In [68]:
def dl_group_by_col(data, columns):

    if isinstance(columns, str):
        columns = [columns]
    
    new_data = pl.DataFrame()

    last_known_data_cols = ['Longitude', 'Latitude','NR_UE_PCI_0','NR_UE_Nbr_PCI_0', 'NR_UE_Nbr_PCI_1',
                           'NR_UE_Nbr_PCI_2', 'NR_UE_Nbr_PCI_3', 'NR_UE_Nbr_PCI_4', 'NR_UE_NACK_Rate_DL_0',
                           'NR_UE_Ack_As_Nack_DL_0', 'NR_UE_MCS_DL_0', 'NR_UE_RB_Num_DL_0',
                           'NR_UE_Modulation_Avg_DL_0', 'NR_UE_RI_DL_0', 'NR_UE_BLER_DL_0',
                           'NR_UE_CCE_AggregationLev_0', 'NR_UE_Power_Tx_PUSCH_0',
                           'NR_UE_Power_Tx_PRACH_0', 'NR_UE_NACK_Rate_UL_0', 'NR_UE_RACH_Attempt',
                           'NR_UE_RACH_OK', 'NR_UE_RACH_Fail', 'NR_UE_RACH_Procedure_Count', 'NR_UE_RRCReEstAttempt',
                           'NR_UE_RRCReEstFail', 'NR_UE_RRCReEst_EndResult', 'NR_UE_RRCConnectionAttempt',
                           'NR_UE_RRCConnectionSetupOk', 'NR_UE_RRCConnectionComplete',
                           'NR_UE_RRCConnectionDrop', 'NR_UE_RRCHOAttempt', 'NR_UE_RRCHOOK',
                           'NR_RRC_MsgType', 'NAS_5GS_MM_MessageType', 'NAS_5GS_SM_MessageType']
    
    mean_cols = ['NR_UE_RSRP_0', 'NR_UE_RSRQ_0', 'NR_UE_SINR_0','NR_UE_Nbr_RSRP_0', 'NR_UE_Nbr_RSRP_1', 'NR_UE_Nbr_RSRP_2',
                'NR_UE_Nbr_RSRP_3', 'NR_UE_Nbr_RSRP_4', 'NR_UE_Nbr_RSRQ_0','NR_UE_Nbr_RSRQ_1', 'NR_UE_Nbr_RSRQ_2', 'NR_UE_Nbr_RSRQ_3',
                'NR_UE_Nbr_RSRQ_4', 'NR_UE_Pathloss_DL_0','NR_UE_Timing_Advance',
                'NR_UE_Throughput_PDCP_DL', 'App_Throughput_DL']
    

    for column in columns:
        if column in last_known_data_cols:
            last_known_data_cols.remove(column)
        if column in mean_cols:
            mean_cols.remove(column)
    
    new_data = data.group_by("Time").agg([
        pl.col(mean_cols).mean(),
        pl.col(last_known_data_cols).forward_fill().last()
    ])

    last_known_data_cols = list(set(last_known_data_cols) - set(["Longitude","Latitude"]))
    mean_cols.append("Time")

    new_data = data.group_by(["Longitude","Latitude"]).agg([
        pl.col(mean_cols).mean(),
        pl.col(last_known_data_cols).drop_nulls().last()
    ])
    
    return new_data

In [69]:
df = dl_group_by_col(dl_data, "Time")
df

Longitude,Latitude,NR_UE_RSRP_0,NR_UE_RSRQ_0,NR_UE_SINR_0,NR_UE_Nbr_RSRP_0,NR_UE_Nbr_RSRP_1,NR_UE_Nbr_RSRP_2,NR_UE_Nbr_RSRP_3,NR_UE_Nbr_RSRP_4,NR_UE_Nbr_RSRQ_0,NR_UE_Nbr_RSRQ_1,NR_UE_Nbr_RSRQ_2,NR_UE_Nbr_RSRQ_3,NR_UE_Nbr_RSRQ_4,NR_UE_Pathloss_DL_0,NR_UE_Timing_Advance,NR_UE_Throughput_PDCP_DL,App_Throughput_DL,Time,NR_UE_CCE_AggregationLev_0,NR_UE_RI_DL_0,NR_UE_RRCConnectionDrop,NR_UE_Modulation_Avg_DL_0,NR_UE_RRCReEstFail,NR_UE_RRCReEst_EndResult,NR_UE_RRCHOOK,NR_UE_Nbr_PCI_0,NR_UE_Nbr_PCI_2,NR_UE_RACH_Fail,NR_UE_Nbr_PCI_3,NR_UE_BLER_DL_0,NR_UE_RACH_OK,NR_UE_RRCReEstAttempt,NR_UE_RRCConnectionSetupOk,NR_UE_RRCConnectionComplete,NR_UE_MCS_DL_0,NR_UE_RACH_Procedure_Count,NR_UE_Power_Tx_PRACH_0,NR_RRC_MsgType,NR_UE_RRCHOAttempt,NR_UE_RB_Num_DL_0,NR_UE_Power_Tx_PUSCH_0,NR_UE_NACK_Rate_UL_0,NR_UE_Nbr_PCI_4,NAS_5GS_MM_MessageType,NR_UE_PCI_0,NAS_5GS_SM_MessageType,NR_UE_Nbr_PCI_1,NR_UE_Ack_As_Nack_DL_0,NR_UE_NACK_Rate_DL_0,NR_UE_RRCConnectionAttempt,NR_UE_RACH_Attempt
f64,f64,f64,f64,f64,f64,str,str,str,str,f64,str,str,str,str,f64,str,f64,str,datetime[ms],str,str,i64,str,i64,str,i64,i64,str,i64,str,str,i64,i64,i64,i64,str,str,str,str,i64,str,f64,f64,str,str,i64,str,str,str,str,i64,i64
29.02779,41.10613,-70.2,-10.4,26.4,null,null,null,null,null,null,null,null,null,null,null,null,96701.48,null,2025-03-14 12:15:11.565,"""LEVEL_1""","""Rank4""",0,"""256QAM""",0,null,0,null,null,0,null,"""11.5""",0,0,0,0,"""24 (64QAM OR 256QAM OR 1024 QA…",null,null,null,0,"""70""",null,null,null,null,76,null,null,"""0""","""11.5""",0,0
29.01833,41.10173,-132.1,-23.9,-10.0,-126.35,null,null,null,null,-19.05,null,null,null,null,157.1,null,8596.905,null,2025-03-14 12:28:27.305,"""LEVEL_16""","""Rank1""",0,"""QPSK""",0,null,0,59,"""68""",0,null,"""11.9""",0,0,0,0,"""6 (QPSK OR 16QAM OR 64QAM / Lo…","""34""",null,""" BCCH-DL_SCH SIB1""",0,"""204""",22.4,25.2,null,null,68,null,"""68""","""27""","""11.9""",0,0
29.0254,41.10781,-92.25,-11.4,7.6,-95.75,null,null,null,null,-14.25,null,null,null,null,111.45,null,96294.435,null,2025-03-14 12:19:26.301,"""LEVEL_2""","""Rank3""",0,"""64QAM""",0,null,0,59,null,0,null,"""9.3""",0,0,0,0,"""21 (64QAM OR 256QAM / Low SE -…",null,null,null,0,"""103""",17.5,0.0,null,null,30,null,null,"""0""","""9.3""",0,0
29.02547,41.10507,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,95841.38,null,2025-03-14 12:20:50.733,null,null,0,null,0,null,0,null,null,0,null,null,0,0,0,0,null,null,null,null,0,null,null,null,null,null,null,null,null,null,null,0,0
29.02555,41.10781,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,2025-03-14 12:24:34.751,"""LEVEL_1""","""Rank4""",0,"""64QAM""",0,null,0,null,null,0,null,"""8.2""",0,0,0,0,"""17 (64QAM OR 256QAM / Low SE -…",null,null,null,0,"""109""",null,null,null,null,null,null,null,"""0""","""8.2""",0,0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
29.01536,41.10588,-121.9,-16.35,-7.7,-122.6,null,null,null,null,-16.45,null,null,null,null,144.65,null,48312.32,null,2025-03-14 12:27:03.318,"""LEVEL_16""","""Rank2""",0,"""16QAM""",0,null,0,30,"""76""",0,null,"""18.5""",0,0,0,0,"""10 (16QAM OR 64QAM / Low SE -Q…",null,null,null,0,"""171""",22.2,14.1,null,null,68,null,"""40""","""1""","""18.5""",0,0
29.03115,41.09978,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,2025-03-14 12:30:37.604,null,null,0,null,0,null,0,null,null,0,null,null,0,0,0,0,null,null,null,null,0,null,null,null,null,null,null,null,null,null,null,0,0
29.02443,41.10785,-73.0,-10.4,30.8,null,null,null,null,null,null,null,null,null,null,92.4,null,98790.98,null,2025-03-14 12:19:17.269,"""LEVEL_1""","""Rank4""",0,"""256QAM""",0,null,0,null,null,0,null,"""3""",0,0,0,0,"""27 (64QAM OR 256QAM / Low SE -…",null,null,null,0,"""25""",17.5,0.0,null,null,30,null,null,"""0""","""3""",0,0


In [87]:
df.sort("Time")

Longitude,Latitude,NR_UE_RSRP_0,NR_UE_RSRQ_0,NR_UE_SINR_0,NR_UE_Nbr_RSRP_0,NR_UE_Nbr_RSRP_1,NR_UE_Nbr_RSRP_2,NR_UE_Nbr_RSRP_3,NR_UE_Nbr_RSRP_4,NR_UE_Nbr_RSRQ_0,NR_UE_Nbr_RSRQ_1,NR_UE_Nbr_RSRQ_2,NR_UE_Nbr_RSRQ_3,NR_UE_Nbr_RSRQ_4,NR_UE_Pathloss_DL_0,NR_UE_Timing_Advance,NR_UE_Throughput_PDCP_DL,App_Throughput_DL,Time,NR_UE_CCE_AggregationLev_0,NR_UE_RI_DL_0,NR_UE_RRCConnectionDrop,NR_UE_Modulation_Avg_DL_0,NR_UE_RRCReEstFail,NR_UE_RRCReEst_EndResult,NR_UE_RRCHOOK,NR_UE_Nbr_PCI_0,NR_UE_Nbr_PCI_2,NR_UE_RACH_Fail,NR_UE_Nbr_PCI_3,NR_UE_BLER_DL_0,NR_UE_RACH_OK,NR_UE_RRCReEstAttempt,NR_UE_RRCConnectionSetupOk,NR_UE_RRCConnectionComplete,NR_UE_MCS_DL_0,NR_UE_RACH_Procedure_Count,NR_UE_Power_Tx_PRACH_0,NR_RRC_MsgType,NR_UE_RRCHOAttempt,NR_UE_RB_Num_DL_0,NR_UE_Power_Tx_PUSCH_0,NR_UE_NACK_Rate_UL_0,NR_UE_Nbr_PCI_4,NAS_5GS_MM_MessageType,NR_UE_PCI_0,NAS_5GS_SM_MessageType,NR_UE_Nbr_PCI_1,NR_UE_Ack_As_Nack_DL_0,NR_UE_NACK_Rate_DL_0,NR_UE_RRCConnectionAttempt,NR_UE_RACH_Attempt
f64,f64,f64,f64,f64,f64,str,str,str,str,f64,str,str,str,str,f64,str,f64,str,datetime[ms],str,str,i64,str,i64,str,i64,i64,str,i64,str,str,i64,i64,i64,i64,str,str,str,str,i64,str,f64,f64,str,str,i64,str,str,str,str,i64,i64
null,null,-87.1,-11.2,7.3,-97.3,null,null,null,null,-16.3,null,null,null,null,null,null,null,null,2025-03-14 12:14:33.285,null,null,0,null,0,null,0,76,null,0,null,null,0,0,0,0,null,null,null,null,0,null,null,null,null,null,48,null,null,null,null,0,0
29.02949,41.10723,-87.173333,-10.9,10.221429,-97.113333,null,null,null,null,-16.186667,null,null,null,null,103.34,null,71794.171176,null,2025-03-14 12:14:36.997,"""LEVEL_2""","""Rank3""",0,"""256QAM""",0,null,0,76,null,0,null,"""8.6""",0,0,0,0,"""23 (64QAM OR 256QAM OR 1024 QA…",null,null,null,0,"""51""",17.5,0.0,null,null,48,null,null,"""0""","""8.6""",0,0
29.02947,41.10722,-88.95,-11.15,8.4,-95.5,null,null,null,null,-16.25,null,null,null,null,104.95,null,97433.6,null,2025-03-14 12:14:41.349,"""LEVEL_2""","""Rank3""",0,"""256QAM""",0,null,0,76,null,0,null,"""9.9""",0,0,0,0,"""23 (64QAM OR 256QAM OR 1024 QA…",null,null,null,0,"""52""",17.5,0.0,null,null,48,null,null,"""0""","""9.9""",0,0
29.02944,41.10724,-89.35,-11.35,7.8,-96.1,null,null,null,null,-15.5,null,null,null,null,106.2,null,97779.195,null,2025-03-14 12:14:42.352,"""LEVEL_2""","""Rank3""",0,"""256QAM""",0,null,0,76,null,0,null,"""8.4""",0,0,0,0,"""23 (64QAM OR 256QAM OR 1024 QA…",null,null,null,0,"""52""",17.5,0.0,null,null,48,null,null,"""0""","""8.4""",0,0
29.02941,41.10725,-91.65,-12.3,4.65,-93.0,null,null,null,null,-13.65,null,null,null,null,107.9,null,98152.98,null,2025-03-14 12:14:43.354,"""LEVEL_2""","""Rank3""",0,"""256QAM""",0,null,0,76,null,0,null,"""9.8""",0,0,0,0,"""23 (64QAM OR 256QAM OR 1024 QA…",null,null,null,0,"""51""",17.5,0.0,null,null,48,null,null,"""0""","""9.8""",0,0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
29.02147,41.10027,-118.55,-13.15,1.1,-122.25,null,null,null,null,-16.25,null,null,null,null,133.65,null,0.545,null,2025-03-14 12:32:07.240,"""LEVEL_16""","""Rank2""",0,"""16QAM""",0,null,0,59,null,0,null,"""0""",0,0,0,0,"""6 (QPSK OR 16QAM OR 64QAM / Lo…",null,null,null,0,"""16""",22.0,8.0,null,null,68,null,null,"""0""","""0""",0,0
29.02146,41.10028,-118.15,-12.65,3.8,-122.45,null,null,null,null,-16.95,null,null,null,null,134.15,null,0.0,null,2025-03-14 12:32:08.307,"""LEVEL_8""","""Rank2""",0,"""16QAM""",0,null,0,59,null,0,null,"""0""",0,0,0,0,"""7 (QPSK OR 16QAM OR 64QAM / Lo…",null,null,null,0,"""16""",22.0,7.9,null,null,68,null,null,"""0""","""0""",0,0
29.02145,41.10029,-118.783333,-13.683333,1.4,-120.883333,null,null,null,null,-15.716667,null,null,null,null,134.133333,null,0.0,null,2025-03-14 12:32:10.300,"""LEVEL_16""","""Rank2""",0,"""16QAM""",0,null,0,59,null,0,null,"""33.3""",0,0,0,0,"""6 (QPSK OR 16QAM OR 64QAM / Lo…",null,null,null,0,"""16""",22.0,8.0,null,null,68,null,null,"""0""","""33.3""",0,0


In [113]:
pl.Config.set_tbl_rows(-1)     # Sınırsız satır
pl.Config.set_tbl_cols(-1)     # Sınırsız sütun
pl.Config.set_tbl_width_chars(2000)  # Geniş ekran
from datetime import datetime, timedelta
for row in df.filter(pl.col("NR_UE_Pathloss_DL_0").is_null()).iter_rows(named=True):
    
    
    target_time = row["Time"]

    # 1 dakika öncesi ve sonrası
    oncesi = target_time - timedelta(seconds=10)
    sonrasi = target_time + timedelta(seconds=10)

    context_data = df.filter(
        (pl.col("Time") >= oncesi) & 
        (pl.col("Time") <= sonrasi)
    ).sort("Time")

    print(context_data["Time","Longitude","Latitude", "NR_UE_PCI_0","NR_UE_Nbr_PCI_1","NR_UE_RSRP_0", "NR_UE_RSRQ_0", "NR_UE_SINR_0","NR_UE_Pathloss_DL_0", 'NR_UE_Nbr_RSRP_1'])

shape: (32, 10)
┌─────────────────────────┬───────────┬──────────┬─────────────┬─────────────────┬──────────────┬──────────────┬──────────────┬─────────────────────┬──────────────────┐
│ Time                    ┆ Longitude ┆ Latitude ┆ NR_UE_PCI_0 ┆ NR_UE_Nbr_PCI_1 ┆ NR_UE_RSRP_0 ┆ NR_UE_RSRQ_0 ┆ NR_UE_SINR_0 ┆ NR_UE_Pathloss_DL_0 ┆ NR_UE_Nbr_RSRP_1 │
│ ---                     ┆ ---       ┆ ---      ┆ ---         ┆ ---             ┆ ---          ┆ ---          ┆ ---          ┆ ---                 ┆ ---              │
│ datetime[ms]            ┆ f64       ┆ f64      ┆ i64         ┆ str             ┆ f64          ┆ f64          ┆ f64          ┆ f64                 ┆ str              │
╞═════════════════════════╪═══════════╪══════════╪═════════════╪═════════════════╪══════════════╪══════════════╪══════════════╪═════════════════════╪══════════════════╡
│ 2025-03-14 12:15:01.667 ┆ 29.0283   ┆ 41.10651 ┆ null        ┆ null            ┆ null         ┆ null         ┆ null         ┆ 93.1       

In [112]:


target_time = pl.datetime(2025, 3, 14, 12, 24, 34, 751000)

oncesi = target_time - timedelta(seconds=10)
sonrasi = target_time + timedelta(seconds=10)

context_data = df.filter(
    (pl.col("Time") >= oncesi) & 
    (pl.col("Time") <= sonrasi)
).sort("Time")

print(context_data)

shape: (30, 53)
┌───────────┬──────────┬──────────────┬──────────────┬──────────────┬──────────────────┬──────────────────┬──────────────────┬──────────────────┬──────────────────┬──────────────────┬──────────────────┬──────────────────┬──────────────────┬──────────────────┬─────────────────────┬──────────────────────┬──────────────────────────┬───────────────────┬─────────────────────────┬────────────────────────────┬───────────────┬─────────────────────────┬───────────────────────────┬────────────────────┬──────────────────────────┬───────────────┬─────────────────┬─────────────────┬─────────────────┬─────────────────┬─────────────────┬───────────────┬───────────────────────┬────────────────────────────┬─────────────────────────────┬─────────────────────────────────┬────────────────────────────┬────────────────────────┬─────────────────────────────────┬────────────────────┬───────────────────┬────────────────────────┬──────────────────────┬─────────────────┬────────────────────────┬─

In [92]:
context_data

Longitude,Latitude,NR_UE_RSRP_0,NR_UE_RSRQ_0,NR_UE_SINR_0,NR_UE_Nbr_RSRP_0,NR_UE_Nbr_RSRP_1,NR_UE_Nbr_RSRP_2,NR_UE_Nbr_RSRP_3,NR_UE_Nbr_RSRP_4,NR_UE_Nbr_RSRQ_0,NR_UE_Nbr_RSRQ_1,NR_UE_Nbr_RSRQ_2,NR_UE_Nbr_RSRQ_3,NR_UE_Nbr_RSRQ_4,NR_UE_Pathloss_DL_0,NR_UE_Timing_Advance,NR_UE_Throughput_PDCP_DL,App_Throughput_DL,Time,NR_UE_CCE_AggregationLev_0,NR_UE_RI_DL_0,NR_UE_RRCConnectionDrop,NR_UE_Modulation_Avg_DL_0,NR_UE_RRCReEstFail,NR_UE_RRCReEst_EndResult,NR_UE_RRCHOOK,NR_UE_Nbr_PCI_0,NR_UE_Nbr_PCI_2,NR_UE_RACH_Fail,NR_UE_Nbr_PCI_3,NR_UE_BLER_DL_0,NR_UE_RACH_OK,NR_UE_RRCReEstAttempt,NR_UE_RRCConnectionSetupOk,NR_UE_RRCConnectionComplete,NR_UE_MCS_DL_0,NR_UE_RACH_Procedure_Count,NR_UE_Power_Tx_PRACH_0,NR_RRC_MsgType,NR_UE_RRCHOAttempt,NR_UE_RB_Num_DL_0,NR_UE_Power_Tx_PUSCH_0,NR_UE_NACK_Rate_UL_0,NR_UE_Nbr_PCI_4,NAS_5GS_MM_MessageType,NR_UE_PCI_0,NAS_5GS_SM_MessageType,NR_UE_Nbr_PCI_1,NR_UE_Ack_As_Nack_DL_0,NR_UE_NACK_Rate_DL_0,NR_UE_RRCConnectionAttempt,NR_UE_RACH_Attempt
f64,f64,f64,f64,f64,f64,str,str,str,str,f64,str,str,str,str,f64,str,f64,str,datetime[ms],str,str,i64,str,i64,str,i64,i64,str,i64,str,str,i64,i64,i64,i64,str,str,str,str,i64,str,f64,f64,str,str,i64,str,str,str,str,i64,i64
29.02561,41.10781,-90.55,-13.4,2.25,-92.5,null,null,null,null,-13.45,null,null,null,null,109.5,null,96130.6,null,2025-03-14 12:24:34.317,"""LEVEL_1""","""Rank4""",0,"""64QAM""",0,null,0,30,null,0,null,"""8.8""",0,0,0,0,"""18 (64QAM OR 256QAM / Low SE -…",null,null,null,0,"""120""",17.8,9.0,null,null,59,null,"""76""","""0""","""8.8""",0,0
29.02555,41.10781,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,2025-03-14 12:24:34.751,"""LEVEL_1""","""Rank4""",0,"""64QAM""",0,null,0,null,null,0,null,"""8.2""",0,0,0,0,"""17 (64QAM OR 256QAM / Low SE -…",null,null,null,0,"""109""",null,null,null,null,null,null,null,"""0""","""8.2""",0,0
29.02549,41.10781,-94.2,-13.6,0.1,-92.3,null,null,null,null,-12.8,null,null,null,null,113.9,null,59860.41,null,2025-03-14 12:24:35.362,"""LEVEL_1""","""Rank4""",0,"""64QAM""",0,null,0,30,null,0,null,"""4""",0,0,0,0,"""17 (64QAM OR 256QAM / Low SE -…",null,null,null,0,"""41""",17.9,9.5,null,null,59,null,"""76""","""0""","""4""",0,0
29.02543,41.10782,-93.0,-13.5,1.2,-94.2,null,null,null,null,-12.9,null,null,null,null,111.6,null,88802.51,null,2025-03-14 12:24:35.649,"""LEVEL_1""","""Rank3""",0,"""256QAM""",0,null,0,30,null,0,null,"""6.9""",0,0,0,0,"""22 (64QAM OR 256QAM / Low SE -…",null,null,null,0,"""37""",18.2,7.0,null,null,59,null,"""76""","""0""","""6.9""",0,0
